In [1]:
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

TensorFlow: 2.20.0
NumPy: 2.0.2


In [2]:
sentences = [
    "I love this movie",
    "This movie is amazing",
    "I really enjoyed this",
    "The movie was fantastic",
    "I like this story",
    "This is a great movie",
    "I am very happy",
    "This was wonderful",
    "I loved the experience",
    "The movie is excellent",
    "I enjoyed watching this",
    "This is really good",
    "Amazing experience",
    "I like it a lot",
    "This movie is awesome",

    "I hate this movie",
    "This movie is terrible",
    "I really disliked this",
    "The movie was boring",
    "I don't like this story",
    "This is a bad movie",
    "I am very disappointed",
    "This was horrible",
    "I hated the experience",
    "The movie is awful",
    "I disliked watching this",
    "This is really bad",
    "Terrible experience",
    "I don't like it",
    "This movie is the worst"
]

labels = np.array([1] * 15 + [0] * 15)

In [3]:
labels

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0])

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)

In [7]:
vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary Size:", vocab_size)
print("Word Index:", tokenizer.word_index)

Vocabulary Size: 39
Word Index: {'this': 1, 'i': 2, 'movie': 3, 'is': 4, 'the': 5, 'really': 6, 'was': 7, 'like': 8, 'experience': 9, 'a': 10, 'amazing': 11, 'enjoyed': 12, 'story': 13, 'am': 14, 'very': 15, 'watching': 16, 'it': 17, 'terrible': 18, 'disliked': 19, "don't": 20, 'bad': 21, 'love': 22, 'fantastic': 23, 'great': 24, 'happy': 25, 'wonderful': 26, 'loved': 27, 'excellent': 28, 'good': 29, 'lot': 30, 'awesome': 31, 'hate': 32, 'boring': 33, 'disappointed': 34, 'horrible': 35, 'hated': 36, 'awful': 37, 'worst': 38}


In [8]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

maxlen = max(len(seq) for seq in sequences)

X = pad_sequences(
    sequences,
    maxlen=maxlen,
    padding="post"
)

print("X shape:", X.shape)
print(X)

X shape: (30, 5)
[[ 2 22  1  3  0]
 [ 1  3  4 11  0]
 [ 2  6 12  1  0]
 [ 5  3  7 23  0]
 [ 2  8  1 13  0]
 [ 1  4 10 24  3]
 [ 2 14 15 25  0]
 [ 1  7 26  0  0]
 [ 2 27  5  9  0]
 [ 5  3  4 28  0]
 [ 2 12 16  1  0]
 [ 1  4  6 29  0]
 [11  9  0  0  0]
 [ 2  8 17 10 30]
 [ 1  3  4 31  0]
 [ 2 32  1  3  0]
 [ 1  3  4 18  0]
 [ 2  6 19  1  0]
 [ 5  3  7 33  0]
 [ 2 20  8  1 13]
 [ 1  4 10 21  3]
 [ 2 14 15 34  0]
 [ 1  7 35  0  0]
 [ 2 36  5  9  0]
 [ 5  3  4 37  0]
 [ 2 19 16  1  0]
 [ 1  4  6 21  0]
 [18  9  0  0  0]
 [ 2 20  8 17  0]
 [ 1  3  4  5 38]]


In [9]:
embedding_dim = 32
num_heads = 4
ff_dim = 64

In [10]:
embedding_layer = tf.keras.layers.Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim
)

x = embedding_layer(X)

print("Embedding shape:", x.shape)

I0000 00:00:1789042335.205049      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789042335.207870      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Embedding shape: (30, 5, 32)


## Transformer Input Pipeline

Raw Text 
   ↓
Tokenization
   ↓
Token IDs
   ↓
Padding
   ↓
Token Embedding

## Vocabulary

Vocabulary is the collection of unique tokens/words known by the tokenizer.

vocab_size = len(tokenizer.word_index) + 1

0 is reserved for padding.

## Embedding

Embedding converts discrete token IDs into dense continuous vectors.

If:

batch = 30
sequence_length = 6
embedding_dim = 32

then:

Embedding output shape = (30, 6, 32)

In [11]:
print(vocab_size)
print(X.shape)
print(x.shape)

39
(30, 5)
(30, 5, 32)


## Positional Encoding

In [12]:
def positional_encoding(maxlen, d_model):
    pos = np.arange(maxlen)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]

    angle = pos / np.power(10000, (2 * (i // 2)) / d_model)

    pe = np.zeros((maxlen, d_model))

    pe[:, 0::2] = np.sin(angle[:, 0::2])
    pe[:, 1::2] = np.cos(angle[:, 1::2])

    return tf.cast(pe, dtype=tf.float32)

In [13]:
pe = positional_encoding(maxlen, embedding_dim)

print(pe.shape)

(5, 32)


In [14]:
x = x + pe

print(x.shape)

(30, 5, 32)


## Self attention

In [15]:
d_model = 32

Wq = tf.keras.layers.Dense(d_model)
Wk = tf.keras.layers.Dense(d_model)
Wv = tf.keras.layers.Dense(d_model)

Q = Wq(x)
K = Wk(x)
V = Wv(x)

print(Q.shape)
print(K.shape)
print(V.shape)

(30, 5, 32)
(30, 5, 32)
(30, 5, 32)


In [16]:
scores = tf.matmul(Q, K, transpose_b=True)

print(scores.shape)

(30, 5, 5)


In [17]:
scores = scores / np.sqrt(d_model)

In [18]:
attention_weights = tf.nn.softmax(scores, axis=-1)

print(attention_weights.shape)

(30, 5, 5)


In [20]:
attention_weights[2]

<tf.Tensor: shape=(5, 5), dtype=float32, numpy=
array([[0.15412803, 0.16951051, 0.18771563, 0.220521  , 0.2681248 ],
       [0.16313298, 0.17289914, 0.1896099 , 0.21840271, 0.25595528],
       [0.18621664, 0.18470177, 0.18970442, 0.20610921, 0.23326796],
       [0.2059608 , 0.19743271, 0.18824092, 0.19151518, 0.21685039],
       [0.20027937, 0.19928105, 0.19108903, 0.19183792, 0.2175126 ]],
      dtype=float32)>

In [21]:
attention_output = tf.matmul(attention_weights, V)

print(attention_output.shape)

(30, 5, 32)


## Multi Head Attention

In [26]:
def multi_head_attention(x, d_model=32, num_heads=4):
    
    head_dim = d_model // num_heads

    Wq = tf.keras.layers.Dense(d_model)
    Wk = tf.keras.layers.Dense(d_model)
    Wv = tf.keras.layers.Dense(d_model)

    Q = Wq(x)
    K = Wk(x)
    V = Wv(x)

    # Split into heads
    Q = tf.reshape(Q, (-1, tf.shape(Q)[1], num_heads, head_dim))
    K = tf.reshape(K, (-1, tf.shape(K)[1], num_heads, head_dim))
    V = tf.reshape(V, (-1, tf.shape(V)[1], num_heads, head_dim))

    # Move heads before sequence dimension
    Q = tf.transpose(Q, [0, 2, 1, 3])
    K = tf.transpose(K, [0, 2, 1, 3])
    V = tf.transpose(V, [0, 2, 1, 3])

    # Attention scores
    scores = tf.matmul(Q, K, transpose_b=True)

    scores = scores / np.sqrt(head_dim)

    weights = tf.nn.softmax(scores, axis=-1)

    # Attention output
    output = tf.matmul(weights, V)

    # Combine heads
    output = tf.transpose(output, [0, 2, 1, 3])

    output = tf.reshape(
        output,
        (-1, tf.shape(output)[1], d_model)
    )

    return output

In [27]:
attention_output = multi_head_attention(x)

print(attention_output.shape)

(30, 5, 32)


## Add and Norm

In [28]:
layer_norm = tf.keras.layers.LayerNormalization()

output = layer_norm(
    x + attention_output
)

print(output.shape)

(30, 5, 32)


## FFN - Feed ForwardNetwork

In [29]:
ffn1 = tf.keras.layers.Dense(64, activation="relu")
ffn2 = tf.keras.layers.Dense(32)

ffn_output = ffn2(ffn1(output))

print(ffn_output.shape)

(30, 5, 32)


In [31]:
layer_norm2 = tf.keras.layers.LayerNormalization()

final_output = layer_norm2(output + ffn_output)

print(final_output.shape)

(30, 5, 32)


## Transformer Encoder

In [32]:
def transformer_encoder(x):

    # 1. Multi-Head Self-Attention
    attention_output = multi_head_attention(
        x,
        d_model=32,
        num_heads=4
    )

    # 2. Add & Norm
    norm1 = tf.keras.layers.LayerNormalization()
    x = norm1(x + attention_output)

    # 3. Feed Forward Network
    ffn1 = tf.keras.layers.Dense(64, activation="relu")
    ffn2 = tf.keras.layers.Dense(32)

    ffn_output = ffn2(ffn1(x))

    # 4. Add & Norm
    norm2 = tf.keras.layers.LayerNormalization()
    output = norm2(x + ffn_output)

    return output

In [33]:
encoder_output = transformer_encoder(x)

print(encoder_output.shape)

(30, 5, 32)


## Stack of 2 encoder block

In [34]:
x = transformer_encoder(x)
x = transformer_encoder(x)

In [35]:
print(x.shape)

(30, 5, 32)


In [36]:
pooling = tf.keras.layers.GlobalAveragePooling1D()

pooled_output = pooling(encoder_output)

print(pooled_output.shape)

(30, 32)


In [37]:
classifier = tf.keras.layers.Dense(1, activation="sigmoid")

predictions = classifier(pooled_output)

print(predictions.shape)

(30, 1)


## Proper Transformer Model

In [38]:
inputs = tf.keras.Input(shape=(maxlen,))

# Embedding
x = tf.keras.layers.Embedding(
    vocab_size,
    32
)(inputs)

# Positional Encoding
pe = positional_encoding(maxlen, 32)
x = x + pe

# Multi-Head Self-Attention
attention = tf.keras.layers.MultiHeadAttention(
    num_heads=4,
    key_dim=8
)(x, x)

# Add & Norm
x = tf.keras.layers.LayerNormalization()(
    x + attention
)

# Feed Forward
ffn = tf.keras.layers.Dense(
    64,
    activation="relu"
)(x)

ffn = tf.keras.layers.Dense(32)(ffn)

# Add & Norm
x = tf.keras.layers.LayerNormalization()(
    x + ffn
)

# Pooling
x = tf.keras.layers.GlobalAveragePooling1D()(x)

# Classification
outputs = tf.keras.layers.Dense(
    1,
    activation="sigmoid"
)(x)

model = tf.keras.Model(inputs, outputs)

In [39]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 5, 32)     │      1,248 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 5, 32)     │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 5, 32)     │      4,224 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 5, 32)     │          0 │ add[0][0],        │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 5, 32)     │         64 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 5, 64)     │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 5, 32)     │      2,080 │ dense_24[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 5, 32)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_25[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 5, 32)     │         64 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 1)         │         33 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,825 (38.38 KB)

 Trainable params: 9,825 (38.38 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [41]:
history = model.fit(
    X,
    labels,
    epochs=30,
    batch_size=4
)

Epoch 1/30
1/8 ━━━━━━━━━━━━━━━━━━━━ 35s 5s/step - accuracy: 0.7500 - loss: 0.6125

I0000 00:00:1789044062.773789     154 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 274ms/step - accuracy: 0.5000 - loss: 0.7216
Epoch 2/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5333 - loss: 0.7017 
Epoch 3/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5000 - loss: 0.7015 
Epoch 4/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5000 - loss: 0.7096 
Epoch 5/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5000 - loss: 0.6882 
Epoch 6/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4667 - loss: 0.6891 
Epoch 7/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5333 - loss: 0.6972 
Epoch 8/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5000 - loss: 0.6830 
Epoch 9/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6000 - loss: 0.6775 
Epoch 10/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5000 - loss: 0.6900 
Epoch 11/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4667 - loss: 0.6835 
Epoch 12/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5000 - loss: 0.6779 
Epoch 13/30

## Testing


In [42]:
def predict_sentiment(sentence):

    # Tokenize
    sequence = tokenizer.texts_to_sequences([sentence])

    # Padding
    sequence = pad_sequences(
        sequence,
        maxlen=maxlen,
        padding="post"
    )

    # Prediction
    prediction = model.predict(sequence, verbose=0)[0][0]

    # Result
    if prediction >= 0.5:
        return "Positive", prediction
    else:
        return "Negative", prediction

In [43]:
predict_sentiment("I love this movie")

('Positive', np.float32(0.9947563))

In [44]:
predict_sentiment("This movie is terrible")

('Negative', np.float32(0.0033158564))

In [45]:
predict_sentiment("This movie was really amazing")

('Positive', np.float32(0.9933975))

In [46]:
predict_sentiment("I hated this movie")

('Negative', np.float32(0.00831931))